# Advanced Deep-Learning Risk Model – Yahoo Finance Pipeline

This notebook extends the base `yahoo_finance_only_investment_risk_pyspark.ipynb` with seven advanced capabilities:

| # | Feature | Benefit |
|---|---------|--------|
| 1 | **TCN + Transformer architectures** | Capture longer / seasonal dependencies |
| 2 | **Multi-asset LSTM with symbol embedding** | Share statistical strength across correlated assets |
| 3 | **Rich features** (RSI-14, MACD, Bollinger Bands) | Improves predictive power |
| 4 | **Optuna hyperparameter search + MLflow** | Systematic tuning with experiment tracking |
| 5 | **GPU acceleration** | Auto-detects CUDA; falls back to CPU |
| 6 | **Flask REST serving + Spark UDF** | Real-time / batch risk scoring |
| 7 | **SHAP explainability** | Trust & compliance for risk-model output |

**Run order:** restart kernel → run Cell 1 → run all remaining cells in sequence.

In [ ]:
# ===========================================================
# SECTION 1 – Environment (run BEFORE any PySpark import)
# ===========================================================
import os, pathlib, sys

HADOOP_HOME = r"C:\Users\21625\Desktop\bigdata\hadoop"
SPARK_TMP   = r"C:\Users\21625\Desktop\bigdata\spark_temp"
JAVA_HOME   = r"C:\Program Files\Microsoft\jdk-17.0.21.21-hotspot"

os.environ["HADOOP_HOME"]      = HADOOP_HOME
os.environ["JAVA_HOME"]        = JAVA_HOME
os.environ["SPARK_LOCAL_DIRS"] = SPARK_TMP
os.environ["PATH"] = HADOOP_HOME + r"\bin;" + os.environ.get("PATH", "")

hadoop_fwd = HADOOP_HOME.replace("\\", "/")
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    f'--driver-java-options "-Dhadoop.home.dir={hadoop_fwd}" pyspark-shell'
)

from pathlib import Path
PROJECT_ROOT = Path.cwd()
MODEL_DIR    = PROJECT_ROOT / "models"
REPORT_DIR   = PROJECT_ROOT / "data" / "reports"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("PYSPARK_SUBMIT_ARGS:", os.environ["PYSPARK_SUBMIT_ARGS"])
print("winutils.exe:", pathlib.Path(HADOOP_HOME, "bin", "winutils.exe").is_file())
print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
# ===========================================================
# SECTION 2 – Install dependencies (run once, then restart kernel)
# ===========================================================
# %pip install -q --upgrade numpy torch torchvision optuna mlflow shap flask requests matplotlib
print("Dependencies already configured in virtual environment.")


In [ ]:
# ===========================================================
# SECTION 3 – Spark session
# ===========================================================
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

# Use pre-downloaded Delta Lake jars (avoids Maven resolution)
jars_dir   = PROJECT_ROOT / 'jars'
delta_jars = ','.join([p.as_uri() for p in jars_dir.glob('*.jar')])

spark = (
    SparkSession.builder
    .appName("dl-advanced-risk")
    .master("local[*]")
    .config("spark.jars", delta_jars)
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)


In [ ]:
# ===========================================================
# SECTION 4 – Load Silver table & feature engineering
# ===========================================================
import pandas as pd
import numpy as np

SILVER_PATH = str(PROJECT_ROOT / "data" / "lakehouse" / "silver" / "market_bars")
silver_df   = spark.read.format("delta").load(SILVER_PATH)
pdf = silver_df.toPandas()
print("Rows loaded:", len(pdf))

# ---- Sort ----
pdf = pdf.sort_values(["symbol", "observation_date"]).reset_index(drop=True)
close_col = "close" if "close" in pdf.columns else "Adj Close"

# ---- Log-return and rolling volatility ----
pdf["log_ret"] = np.log(pdf[close_col] / pdf[close_col].shift(1))
pdf["vol_10"]  = (
    pdf.groupby("symbol")["log_ret"]
       .transform(lambda s: s.rolling(10, min_periods=1).std())
)

# ---- Technical indicators (pure pandas) ----
def compute_rsi(series, period=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).ewm(alpha=1/period, adjust=False).mean()
    loss  = (-delta.clip(upper=0)).ewm(alpha=1/period, adjust=False).mean()
    rs    = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def compute_macd_signal(series, fast=12, slow=26, signal=9):
    ema_fast   = series.ewm(span=fast, adjust=False).mean()
    ema_slow   = series.ewm(span=slow, adjust=False).mean()
    macd_line  = ema_fast - ema_slow
    return macd_line.ewm(span=signal, adjust=False).mean()

def compute_bb_width(series, period=20):
    sma  = series.rolling(period, min_periods=1).mean()
    std  = series.rolling(period, min_periods=1).std().replace(0, np.nan)
    return (2 * std) / sma

def per_symbol(fn, df, col):
    return df.groupby("symbol")[col].transform(fn)

pdf["rsi_14"]      = per_symbol(compute_rsi,         pdf, close_col)
pdf["macd_signal"] = per_symbol(compute_macd_signal, pdf, close_col)
pdf["bb_width"]    = per_symbol(compute_bb_width,    pdf, close_col)

# ---- Target ----
pdf["target"] = (
    pdf.groupby("symbol")["vol_10"]
       .shift(-1)
       .fillna(0)
)

# ---- Fill NaN ----
feature_cols = ["log_ret", "vol_10", "rsi_14", "macd_signal", "bb_width"]
pdf[feature_cols] = pdf[feature_cols].fillna(0)

# ---- Symbol-to-int mapping (for embedding layer) ----
symbols   = sorted(pdf["symbol"].unique())
sym2idx   = {s: i for i, s in enumerate(symbols)}
num_syms  = len(symbols)
pdf["sym_idx"] = pdf["symbol"].map(sym2idx)

# ---- Train / val split by date ----
split_date = pd.Timestamp("2025-01-01")
date_col   = "observation_date" if "observation_date" in pdf.columns else "date"
train_mask = pd.to_datetime(pdf[date_col]) < split_date

X      = pdf[feature_cols].values.astype(np.float32)
y      = pdf["target"].values.astype(np.float32)
sym_id = pdf["sym_idx"].values.astype(np.int64)

X_train, X_val     = X[train_mask],      X[~train_mask]
y_train, y_val     = y[train_mask],      y[~train_mask]
s_train, s_val     = sym_id[train_mask], sym_id[~train_mask]

print(f"Features: {feature_cols}")
print(f"Symbols: {num_syms}  |  Train: {X_train.shape[0]}  |  Val: {X_val.shape[0]}")


In [ ]:
# ===========================================================
# SECTION 5 – GPU device
# ===========================================================
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ===========================================================
# SECTION 6 – Model architectures
# ===========================================================
import torch.nn as nn

INPUT_DIM = len(feature_cols)   # 5

# ------------------------------------------------------------------
# A. Temporal Convolutional Network (TCN)
# ------------------------------------------------------------------
class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, dilation=1, dropout=0.1):
        super().__init__()
        pad = (kernel - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel,
                              padding=pad, dilation=dilation)
        self.relu    = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.residual = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self._pad = pad

    def forward(self, x):
        # causal: trim future padding
        out = self.conv(x)
        if self._pad > 0:
            out = out[:, :, :-self._pad]
        out = self.dropout(self.relu(out))
        return out + self.residual(x)

class VolatilityTCN(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, channels=32):
        super().__init__()
        self.net = nn.Sequential(
            TCNBlock(input_dim, channels, dilation=1),
            TCNBlock(channels,  channels, dilation=2),
            TCNBlock(channels,  channels, dilation=4),
        )
        self.fc = nn.Linear(channels, 1)

    def forward(self, x):
        # x: (batch, seq, feat) -> (batch, feat, seq)
        out = self.net(x.permute(0, 2, 1))
        return self.fc(out[:, :, -1]).squeeze(-1)

# ------------------------------------------------------------------
# B. Transformer Risk Model
# ------------------------------------------------------------------
class VolatilityTransformer(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, d_model=32, nhead=4, num_layers=2):
        super().__init__()
        self.proj    = nn.Linear(input_dim, d_model)
        enc_layer    = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=64, batch_first=True, dropout=0.1
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.fc      = nn.Linear(d_model, 1)

    def forward(self, x):
        # x: (batch, seq, feat)
        out = self.encoder(self.proj(x))
        return self.fc(out[:, -1, :]).squeeze(-1)

# ------------------------------------------------------------------
# C. Multi-Asset LSTM with symbol embedding
# ------------------------------------------------------------------
class MultiAssetLSTM(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden=32, layers=2,
                 num_symbols=50, embed_dim=8):
        super().__init__()
        self.embed = nn.Embedding(num_symbols, embed_dim)
        lstm_drop  = 0.1 if layers > 1 else 0.0
        self.lstm  = nn.LSTM(input_dim + embed_dim, hidden,
                             num_layers=layers, batch_first=True,
                             dropout=lstm_drop)
        self.fc    = nn.Linear(hidden, 1)

    def forward(self, x, sym_idx):
        # x: (batch, seq, feat)  sym_idx: (batch,)
        emb  = self.embed(sym_idx).unsqueeze(1).expand(-1, x.size(1), -1)
        out, _ = self.lstm(torch.cat([x, emb], dim=-1))
        return self.fc(out[:, -1, :]).squeeze(-1)

print("Model classes defined: VolatilityTCN | VolatilityTransformer | MultiAssetLSTM")


In [ ]:
# ===========================================================
# SECTION 7 – Training utility
# ===========================================================
from torch.utils.data import TensorDataset, DataLoader

def make_loader(X, y, sym=None, batch_size=512, shuffle=True):
    tensors = [torch.from_numpy(X).unsqueeze(1),   # (N,1,feat)
               torch.from_numpy(y)]
    if sym is not None:
        tensors.insert(1, torch.from_numpy(sym))
    return DataLoader(TensorDataset(*tensors),
                      batch_size=batch_size, shuffle=shuffle)

def train_model(model, train_dl, val_X, val_y, val_sym=None,
                epochs=15, lr=1e-3, dev=device):
    model     = model.to(dev)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_rmse = float("inf")

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        for batch in train_dl:
            if len(batch) == 3:
                xb, symb, yb = [t.to(dev) for t in batch]
                pred = model(xb, symb)
            else:
                xb, yb = [t.to(dev) for t in batch]
                pred = model(xb)
            optimizer.zero_grad()
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * xb.size(0)

        model.eval()
        with torch.no_grad():
            vx = torch.from_numpy(val_X).unsqueeze(1).to(dev)
            vy = torch.from_numpy(val_y).to(dev)
            if val_sym is not None:
                vs  = torch.from_numpy(val_sym).to(dev)
                vp  = model(vx, vs)
            else:
                vp = model(vx)
            rmse = torch.sqrt(criterion(vp, vy)).item()

        best_rmse = min(best_rmse, rmse)
        print(f"epoch {epoch:02d}  "
              f"train={epoch_loss/len(train_dl.dataset):.6f}  "
              f"val_rmse={rmse:.6f}")

    return model, best_rmse

print("train_model() ready.")


In [ ]:
# ===========================================================
# SECTION 8 – Train all three models & compare
# ===========================================================
results = {}

# A – TCN
print("\n--- Training VolatilityTCN ---")
tcn_dl  = make_loader(X_train, y_train)
tcn_model, tcn_rmse = train_model(
    VolatilityTCN(INPUT_DIM),
    tcn_dl, X_val, y_val, epochs=15
)
results["TCN"] = tcn_rmse

# B – Transformer
print("\n--- Training VolatilityTransformer ---")
tf_dl   = make_loader(X_train, y_train)
tf_model, tf_rmse = train_model(
    VolatilityTransformer(INPUT_DIM),
    tf_dl, X_val, y_val, epochs=15
)
results["Transformer"] = tf_rmse

# C – Multi-Asset LSTM
print("\n--- Training MultiAssetLSTM ---")
ma_dl   = make_loader(X_train, y_train, sym=s_train)
ma_model, ma_rmse = train_model(
    MultiAssetLSTM(INPUT_DIM, num_symbols=num_syms),
    ma_dl, X_val, y_val, val_sym=s_val, epochs=15
)
results["MultiAssetLSTM"] = ma_rmse

print("\n=== Validation RMSE comparison ===")
for name, rmse in sorted(results.items(), key=lambda kv: kv[1]):
    print(f"  {name:<20s}: {rmse:.6f}")


In [ ]:
# ===========================================================
# SECTION 9 – Optuna hyperparameter search + MLflow
# ===========================================================
import optuna, mlflow
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
optuna.logging.set_verbosity(optuna.logging.WARNING)

mlflow.set_tracking_uri("sqlite:///" + str(PROJECT_ROOT / "mlflow.db"))
mlflow.set_experiment("yahoo_finance_risk_dl")

class SmallLSTM(nn.Module):
    def __init__(self, input_dim, hidden, layers):
        super().__init__()
        drop = 0.1 if layers > 1 else 0.0
        self.lstm = nn.LSTM(input_dim, hidden, num_layers=layers,
                            batch_first=True, dropout=drop)
        self.fc   = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

def objective(trial):
    hidden = trial.suggest_categorical("hidden", [32, 64, 128])
    lr     = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    layers = trial.suggest_int("layers", 1, 3)
    epochs = trial.suggest_int("epochs", 8, 20)

    model  = SmallLSTM(INPUT_DIM, hidden, layers)
    dl     = make_loader(X_train, y_train, batch_size=512)
    _, rmse = train_model(model, dl, X_val, y_val, epochs=epochs, lr=lr)

    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}"):
        mlflow.log_params(trial.params)
        mlflow.log_metric("val_rmse", rmse)

    return rmse

with mlflow.start_run(run_name="optuna_search"):
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=15, show_progress_bar=True)
    best = study.best_params
    mlflow.log_params({"best_" + k: v for k, v in best.items()})
    mlflow.log_metric("best_val_rmse", study.best_value)

print("\nBest params:", best)
print("Best RMSE  :", study.best_value)

try:
    ax = optuna.visualization.matplotlib.plot_param_importances(study)
    ax.figure.tight_layout()
    ax.figure.savefig(str(REPORT_DIR / "optuna_param_importance.png"), dpi=150)
    print("Param-importance plot saved.")
except Exception as e:
    print("Could not plot param importance:", e)


In [ ]:
# ===========================================================
# SECTION 10 – Save all checkpoints
# ===========================================================
torch.save(tcn_model.state_dict(),
           str(MODEL_DIR / "vol_tcn.pt"))
torch.save(tf_model.state_dict(),
           str(MODEL_DIR / "vol_transformer.pt"))
torch.save(ma_model.state_dict(),
           str(MODEL_DIR / "vol_multi_asset_lstm.pt"))

# Also log best TCN to MLflow
import mlflow.pytorch
with mlflow.start_run(run_name="best_tcn"):
    mlflow.log_metric("val_rmse", tcn_rmse)
    mlflow.pytorch.log_model(tcn_model.cpu(), "tcn_model")

print("Checkpoints saved to:", MODEL_DIR)
for f in MODEL_DIR.iterdir():
    print(" ", f.name)


In [ ]:
# ===========================================================
# SECTION 11 – SHAP explainability
# ===========================================================
import shap
import matplotlib.pyplot as plt

explain_model = tcn_model.cpu().eval()

def model_predict_numpy(x_np):
    with torch.no_grad():
        t = torch.from_numpy(x_np.astype(np.float32)).unsqueeze(1)
        return explain_model(t).numpy()

# Use a small background dataset
bg_size   = min(100, len(X_train))
val_size  = min(50,  len(X_val))
background = X_train[:bg_size]

print("Running SHAP KernelExplainer (may take a few minutes)...")
explainer = shap.KernelExplainer(model_predict_numpy, background)
shap_vals = explainer.shap_values(X_val[:val_size], nsamples=100)

plt.figure(figsize=(8, 4))
shap.summary_plot(shap_vals, X_val[:val_size],
                  feature_names=feature_cols, show=False)
plt.tight_layout()
shap_path = str(REPORT_DIR / "shap_summary.png")
plt.savefig(shap_path, dpi=150)
plt.show()
print("SHAP summary saved to:", shap_path)


In [ ]:
# ===========================================================
# SECTION 12 – Flask REST server + Spark pandas_udf inference
# ===========================================================
import threading, json
import flask
import requests as req
from pyspark.sql.functions import pandas_udf
import pandas as pd

# ---- A: start Flask server in a background daemon thread ----
serve_app    = flask.Flask("risk_model_server")
_cache = {}

def _get_model():
    if "tcn" not in _cache:
        m = VolatilityTCN(INPUT_DIM)
        m.load_state_dict(
            torch.load(str(MODEL_DIR / "vol_tcn.pt"),
                       map_location="cpu", weights_only=False)
        )
        m.eval()
        _cache["tcn"] = m
    return _cache["tcn"]

@serve_app.route("/predict", methods=["POST"])
def predict():
    data     = flask.request.get_json(force=True)
    features = np.array(data["features"], dtype=np.float32)  # (N, feat)
    inp      = torch.from_numpy(features).unsqueeze(1)        # (N, 1, feat)
    with torch.no_grad():
        out = _get_model()(inp).squeeze().tolist()
    if isinstance(out, float):
        out = [out]
    return flask.jsonify({"predicted_vol": out})

@serve_app.route("/health")
def health():
    return flask.jsonify({"status": "ok"})

SERVER_PORT = 5050
t = threading.Thread(
    target=lambda: serve_app.run(port=SERVER_PORT, debug=False, use_reloader=False),
    daemon=True
)
t.start()
import time; time.sleep(1.5)  # let Flask start

health_resp = req.get(f"http://localhost:{SERVER_PORT}/health", timeout=5)
print("Flask server:", health_resp.json())

# ---- B: quick smoke-test of the endpoint ----
sample     = X_val[:4].tolist()
resp       = req.post(f"http://localhost:{SERVER_PORT}/predict",
                      json={"features": sample}, timeout=10)
print("Sample prediction:", resp.json())

# ---- C: Spark pandas_udf that calls the REST endpoint ----
@pandas_udf("double")
def dl_risk_score(*cols: pd.Series) -> pd.Series:
    feat  = np.stack([c.values for c in cols], axis=1).tolist()
    r     = req.post(f"http://localhost:{SERVER_PORT}/predict",
                     json={"features": feat}, timeout=30)
    return pd.Series(r.json()["predicted_vol"])

# ---- D: apply UDF to the Gold table and write to dl_risk_scores ----
GOLD_PATH = str(PROJECT_ROOT / "data" / "lakehouse" / "gold" / "investment_risk_features")
gold_df   = spark.read.format("delta").load(GOLD_PATH)

# Add log_ret if missing (may already exist from previous run)
if "log_ret" not in gold_df.columns:
    w_sym   = Window.partitionBy("symbol").orderBy("observation_date")
    gold_df = gold_df.withColumn(
        "log_ret",
        F.log(F.col("close") / F.lag("close").over(w_sym))
    ).na.fill({"log_ret": 0.0})

# Add placeholder indicator columns if absent
for col_name in ["vol_10", "rsi_14", "macd_signal", "bb_width"]:
    if col_name not in gold_df.columns:
        gold_df = gold_df.withColumn(col_name, F.lit(0.0))

scored = gold_df.withColumn(
    "dl_predicted_vol",
    dl_risk_score("log_ret", "vol_10", "rsi_14", "macd_signal", "bb_width")
)

DL_GOLD = str(PROJECT_ROOT / "data" / "lakehouse" / "gold" / "dl_risk_scores")
scored.write.format("delta").mode("overwrite").save(DL_GOLD)

print("DL risk scores written to:", DL_GOLD)
scored.select("symbol", "observation_date", "close", "dl_predicted_vol").show(10)


## Summary

| Component | Implementation | Output |
|-----------|---------------|--------|
| **TCN** | `VolatilityTCN` – dilated causal Conv1D (3 blocks, channels=32) | `models/vol_tcn.pt` |
| **Transformer** | `VolatilityTransformer` – `nn.TransformerEncoder` (2L, 4H) | `models/vol_transformer.pt` |
| **Multi-asset LSTM** | `MultiAssetLSTM` – `nn.Embedding(n_sym, 8)` + LSTM | `models/vol_multi_asset_lstm.pt` |
| **Rich features** | `log_ret`, `vol_10`, RSI-14, MACD-signal, BB-width (pure pandas) | Used in all models |
| **GPU** | `torch.device("cuda" if cuda else "cpu")` | Auto-detected |
| **Optuna + MLflow** | 15-trial search over hidden / lr / layers / epochs | `data/reports/optuna_param_importance.png` |
| **SHAP** | `KernelExplainer` on TCN, summary plot | `data/reports/shap_summary.png` |
| **Flask REST** | Background thread on `:5050`, `/predict` and `/health` routes | Live during notebook session |
| **Spark UDF** | `dl_risk_score` calls Flask endpoint per batch | `gold/dl_risk_scores` Delta table |

### View MLflow runs
```bash
mlflow ui
# open http://localhost:5000
```

### Next steps
- Feed real macro data (FRED API: interest rates, VIX) into the feature pipeline.
- Replace placeholder indicator values in Spark UDF with Spark-native rolling computations.
- Deploy the Flask server as a Docker container or Azure/GCP Function for production serving.
- Add walk-forward backtesting to evaluate forecast quality over time.